# Rawr-AGI-2 Nemo

Fork of the proven Launch B baseline (`kaggle_notebook/`, public LB 29.31): environment setup, `arc_loader.py`, `arc_decoder.py`, `symbolic_size.py`, `arc_solver.py`, `starter.py`, the submission cell, and the Leg C merge cell are byte-identical to that run.

**The only change:** Leg C's induction engine is now NVIDIA Nemotron-3.5-Lightning-30B-A3B-NVFP4 via vLLM (`nemotron_induction.py`) instead of Qwen2.5-Coder-7B. It reuses the same AST-sandboxed, demo-verified-only candidate acceptance as the tested probe core, and writes `/kaggle/working/induction_results.json` in the exact shape the (unmodified) Leg C merge cell already consumes.

This **is** a submission notebook: it writes `submission.json` from the base pipeline regardless of whether Nemotron induction verifies anything, then the merge cell promotes any verified Nemotron outputs to attempt_1. If Nemotron verifies nothing, this is byte-equivalent to the Launch B baseline.

vLLM installs into an isolated site-packages directory; its PYTHONPATH is passed only to the induction subprocess, never to the base (unsloth) pipeline's subprocess, to avoid a torch-ABI conflict between the two runtimes.


In [ ]:
# Keep the original global 10-minute submission/write buffer.
import time
global_end_time = time.time() + 12 * 3600 - 600


In [ ]:
# Preserve the baseline environment workaround.
!pip uninstall -y tensorflow


In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
from math import exp, log
import numpy as np


def _log_mean_exp(values):
    """Stable log(mean(exp(values))) for a non-empty finite iterable."""
    values = [float(value) for value in values]
    if not values:
        raise ValueError("log mean exp requires at least one value")
    pivot = max(values)
    return pivot + log(sum(exp(value - pivot) for value in values) / len(values))

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    # Filesystem/sample insertion order is not a semantic tie-breaker.  Use a
    # canonical grid tuple so replaying the same cache is deterministic.
    scores = sorted(scores, key=lambda x: (-x[0], hashable(x[1])))
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


def getter_log_evidence(guesses):
    """Rank one exact-output class by conservative nuisance-marginal evidence.

    View transforms and repeated samples are treated as exchangeable evidence,
    so each mean is taken in probability space before combining the beam and
    augmentation likelihood terms.  This avoids raw support-count inflation.
    It is an ablation only: the model scores are not calibrated probabilities.
    """
    if not guesses:
        return float("-inf")
    beam_log_marginal = _log_mean_exp(
        -float(guess["beam_score"]) for guess in guesses
    )
    augmentation_log_marginals = (
        _log_mean_exp(-float(score) for score in guess["score_aug"])
        for guess in guesses
    )
    return beam_log_marginal + _log_mean_exp(augmentation_log_marginals)


def score_log_evidence(guesses):
    """Return exact-output classes ranked by opt-in log evidence."""
    return score_sum(guesses, getter_log_evidence)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
    score_log_evidence,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")


In [ ]:
%%writefile symbolic_size.py
"""Self-contained falsification-based output-size predictor (paranoid preset).

Embedded copy of the repo's symbolic/size_predictor.py + grid_utils.py,
flattened into one dependency-free module for the offline Kaggle runtime.
Every rule is fitted against ALL demonstration pairs and fires only when it
explains every one exactly; all fired rules must agree or the predictor
abstains. The paranoid preset measured 100% precision (zero size errors) on
both the 1000-task training set and the 120-task evaluation set, firing on
~63% of evaluation test outputs.

Public API: predict_size_paranoid(train_pairs, test_input) -> (h, w) | None
where train_pairs is a list of (input_grid, output_grid) tuples.
"""

from collections import Counter, deque
from fractions import Fraction

MAX_DIM = 30


def dims(grid):
    return (len(grid), len(grid[0]) if grid else 0)


def palette(grid):
    return frozenset(c for row in grid for c in row)


def color_counts(grid):
    counts = Counter()
    for row in grid:
        counts.update(row)
    return counts


def most_common_color(grid):
    counts = color_counts(grid)
    best = max(counts.items(), key=lambda kv: (kv[1], -kv[0]))
    return best[0]


def bbox_dims(grid, bg):
    rows = [r for r, row in enumerate(grid) if any(c != bg for c in row)]
    if not rows:
        return None
    cols = [c for c in range(len(grid[0]))
            if any(grid[r][c] != bg for r in range(len(grid)))]
    return (rows[-1] - rows[0] + 1, cols[-1] - cols[0] + 1)


def connected_components(grid, connectivity, bg):
    h, w = dims(grid)
    if connectivity == 4:
        steps = ((-1, 0), (1, 0), (0, -1), (0, 1))
    else:
        steps = ((-1, -1), (-1, 0), (-1, 1), (0, -1),
                 (0, 1), (1, -1), (1, 0), (1, 1))
    seen = [[False] * w for _ in range(h)]
    comps = []
    for r0 in range(h):
        for c0 in range(w):
            if seen[r0][c0] or grid[r0][c0] == bg:
                continue
            comp = []
            queue = deque([(r0, c0)])
            seen[r0][c0] = True
            while queue:
                r, c = queue.popleft()
                comp.append((r, c))
                for dr, dc in steps:
                    nr, nc = r + dr, c + dc
                    if (0 <= nr < h and 0 <= nc < w
                            and not seen[nr][nc] and grid[nr][nc] != bg):
                        seen[nr][nc] = True
                        queue.append((nr, nc))
            comps.append(comp)
    return comps


def component_bbox_dims(comp):
    rs = [r for r, _ in comp]
    cs = [c for _, c in comp]
    return (max(rs) - min(rs) + 1, max(cs) - min(cs) + 1)


def valid_size(size):
    if size is None:
        return False
    h, w = size
    return (isinstance(h, int) and isinstance(w, int)
            and 1 <= h <= MAX_DIM and 1 <= w <= MAX_DIM)


class FittedRule:

    def __init__(self, name, predict_fn):
        self.name = name
        self._predict = predict_fn

    def predict(self, test_input):
        size = self._predict(test_input)
        return size if valid_size(size) else None


def _fit_constant(pairs):
    sizes = {dims(o) for _, o in pairs}
    if len(sizes) != 1:
        return None
    size = next(iter(sizes))
    return lambda test: size


def _fit_same_as_input(pairs):
    if all(dims(i) == dims(o) for i, o in pairs):
        return lambda test: dims(test)
    return None


def _fit_transpose(pairs):
    if all(dims(o) == (dims(i)[1], dims(i)[0]) for i, o in pairs):
        if any(dims(i)[0] != dims(i)[1] for i, _ in pairs):
            return lambda test: (dims(test)[1], dims(test)[0])
    return None


def _fit_ratio(pairs):
    rh = {Fraction(dims(o)[0], dims(i)[0]) for i, o in pairs}
    rw = {Fraction(dims(o)[1], dims(i)[1]) for i, o in pairs}
    if len(rh) != 1 or len(rw) != 1:
        return None
    fh, fw = next(iter(rh)), next(iter(rw))
    if fh == 1 and fw == 1:
        return None

    def predict(test):
        h, w = dims(test)
        nh, nw = Fraction(h) * fh, Fraction(w) * fw
        if nh.denominator != 1 or nw.denominator != 1:
            return None
        return (int(nh), int(nw))

    return predict


def _fit_affine(pairs):
    ch = {dims(o)[0] - dims(i)[0] for i, o in pairs}
    cw = {dims(o)[1] - dims(i)[1] for i, o in pairs}
    if len(ch) != 1 or len(cw) != 1:
        return None
    c, d = next(iter(ch)), next(iter(cw))
    if c == 0 and d == 0:
        return None
    return lambda test: (dims(test)[0] + c, dims(test)[1] + d)


def _make_bbox_fitter(bg_mode):

    def bbox_of(grid):
        bg = most_common_color(grid) if bg_mode == "mode" else 0
        return bbox_dims(grid, bg)

    def fit(pairs):
        for i, o in pairs:
            if bbox_of(i) != dims(o):
                return None
        if all(bbox_of(i) == dims(i) for i, _ in pairs):
            return None
        return bbox_of

    return fit


def _make_object_fitter(connectivity, largest):

    def target_bbox(grid):
        bg = most_common_color(grid)
        comps = connected_components(grid, connectivity, bg)
        if not comps:
            return None
        key = max if largest else min
        best = key(len(c) for c in comps)
        boxes = {component_bbox_dims(c) for c in comps if len(c) == best}
        if len(boxes) != 1:
            return None
        return next(iter(boxes))

    def fit(pairs):
        for i, o in pairs:
            if target_bbox(i) != dims(o):
                return None
        return target_bbox

    return fit


_ALL_RULES = [
    ("same_as_input", _fit_same_as_input),
    ("constant", _fit_constant),
    ("ratio", _fit_ratio),
    ("affine_offset", _fit_affine),
    ("transpose", _fit_transpose),
    ("bbox_nonbg", _make_bbox_fitter("mode")),
    ("bbox_nonzero", _make_bbox_fitter("zero")),
    ("largest_obj_4", _make_object_fitter(4, True)),
    ("largest_obj_8", _make_object_fitter(8, True)),
    ("smallest_obj_4", _make_object_fitter(4, False)),
    ("smallest_obj_8", _make_object_fitter(8, False)),
]

# Paranoid preset: color-count rules removed entirely; object/bbox rules need
# >=4 demos to fire at all; `constant` needs >=4 demos to stand alone.
PARANOID_MIN_DEMOS = {
    "bbox_nonbg": 4,
    "bbox_nonzero": 4,
    "largest_obj_4": 4,
    "largest_obj_8": 4,
    "smallest_obj_4": 4,
    "smallest_obj_8": 4,
}
PARANOID_MIN_DEMOS_SOLO = {
    "constant": 4,
}


def predict_size_paranoid(train_pairs, test_input):
    """Predicted (height, width) of the test output, or None (abstain)."""
    n = len(train_pairs)
    candidates = []
    for name, fitter in _ALL_RULES:
        if n < PARANOID_MIN_DEMOS.get(name, 1):
            continue
        fn = fitter(train_pairs)
        if fn is None:
            continue
        size = FittedRule(name, fn).predict(test_input)
        if size is not None:
            candidates.append((name, size))

    strong = [(name, s) for name, s in candidates
              if n >= PARANOID_MIN_DEMOS_SOLO.get(name, 1)]
    if not strong:
        return None

    distinct = {s for _, s in candidates}
    if len(distinct) == 1:
        return strong[0][1]
    return None

In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import json
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15

# Symbolic decode cap: when the falsification-based size predictor fires
# (paranoid preset: zero measured size errors across the 1000 training and
# 120 evaluation tasks), cap DFS generation at the predicted grid's token
# count instead of the 30x30 worst case (931 tokens). False -> exactly the
# baseline behavior.
SIZE_CAP_TOKENS = True

from symbolic_size import predict_size_paranoid


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


# Minimal performance patch: preserve the baseline beam set and ranking, but transfer
# only the 12 ARC-token NLL values to CPU instead of every Qwen vocabulary logit.
_ARC_TOKEN_ID_CACHE = {}


def _arc_token_ids(device):
    key = str(device)
    token_ids = _ARC_TOKEN_ID_CACHE.get(key)
    if token_ids is None:
        token_ids = torch.tensor(ARC_TOKENS, dtype=torch.long, device=device)
        _ARC_TOKEN_ID_CACHE[key] = token_ids
    return token_ids


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    # Algebraically identical to: scores - logits.float().cpu().log_softmax(-1),
    # restricted to the same ARC_TOKENS used by the baseline DFS loop.
    logits_f = logits.float()
    token_ids = _arc_token_ids(logits.device)
    arc_logits = logits_f.index_select(-1, token_ids)
    nll = (
        torch.as_tensor(scores, dtype=torch.float32, device=logits.device).view(n, 1)
        + torch.logsumexp(logits_f, dim=-1, keepdim=True)
        - arc_logits
    ).cpu()

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for token_idx, t in enumerate(ARC_TOKENS):
            score = nll[i, token_idx].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)

    # Keep logits on GPU and gather only the target-token scores. KV cache is not
    # consumed by teacher-forced scoring, so disabling it removes redundant writes.
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=False)
    batch_logits = outputs.logits.float()
    batch_log_norm = torch.logsumexp(batch_logits, dim=-1)
    result = []
    for row_id, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_length = len(answer_tokens)
        positions = torch.arange(
            query_length - 1,
            query_length - 1 + answer_length,
            device=model.device,
        )
        target_tokens = torch.tensor(answer_tokens, device=model.device, dtype=torch.long)
        answer_log_probs = (
            batch_logits[row_id, positions, target_tokens]
            - batch_log_norm[row_id, positions]
        )
        result.append(-answer_log_probs.sum().item())
    return result


def worker(rank, queue, end_time):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    if SIZE_CAP_TOKENS:
        with open(test_path, "r") as f:
            raw_challenges = json.load(f)

    dir_outputs = "/kaggle/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    # The starter enqueues one sentinel per worker.  ``Queue.empty()`` is not
    # a synchronization primitive for a managed multiprocessing queue: a
    # worker can observe a transient empty state and exit before a queued task
    # becomes visible.  Block on the sentinel protocol instead.
    while True:

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break
        
        start_time = time.time()
        
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=1)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )

            stats = trainer.train()

            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

            del trainer

        model = FastLanguageModel.for_inference(model)
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        # Predict output sizes once per task from the ORIGINAL (unaugmented)
        # demo pairs; keyed by test_id. Any failure degrades to "no cap".
        size_caps = {}
        if SIZE_CAP_TOKENS:
            try:
                raw_task = raw_challenges[key]
                raw_pairs = [(p["input"], p["output"]) for p in raw_task["train"]]
                for test_id, test_pair in enumerate(raw_task["test"]):
                    predicted = predict_size_paranoid(raw_pairs, test_pair["input"])
                    if predicted is not None:
                        size_caps[str(test_id)] = predicted
                if size_caps:
                    print(f"[Rank {rank}] size caps for {key}: {size_caps}")
            except Exception as exc:
                print(f"[Rank {rank}] size predictor failed for {key}: {exc}")
                size_caps = {}

        eval_ds = puzzle_ds_multi.augment(n=2, seed=2)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            # 0: permute x 2
            # 4: rot90.rot90.permute x 2
            batch = []
            for offset in [0, 4]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 2: permute.rot90 x 2
            # 6: rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [2, 6]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
        for test_id, subkeys in test_id_to_subkeys.items():
            # 8: transpose.permute x 2
            # 12: transpose.rot90.rot90.permute x 2
            batch = []
            for offset in [8, 12]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 10: transpose.rot90.permute x 2
            # 14: transpose.rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [10, 14]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)

        with torch.inference_mode():
                
            known_scores = {}

            for subkeys in batches:

                spend_time = time.time() - start_time
                if spend_time > 1200 or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                # Per-batch decode cap. Every subkey in a batch shares one
                # test_id; rot90/transpose each swap height/width, so the cap
                # is computed per augmented view (odd swap parity -> swapped
                # dims). Grid token count = h*w digits + h-1 newlines + EOS
                # = h*w + h, plus 2 slack. Missing prediction -> no cap.
                batch_max_new_tokens = max_new_tokens
                if SIZE_CAP_TOKENS:
                    caps = []
                    for subkey in subkeys:
                        predicted = size_caps.get(subkey.split(".")[0].split("_")[1])
                        if predicted is None:
                            caps = None
                            break
                        cap_h, cap_w = predicted
                        swaps = sum(op in ("rot90", "transpose") for op in subkey.split(".")[1:])
                        if swaps % 2:
                            cap_h, cap_w = cap_w, cap_h
                        caps.append(cap_h * cap_w + cap_h + 2)
                    if caps:
                        batch_max_new_tokens = min(max_new_tokens, max(caps))

                dfs_result = inference_turbo_dfs(model, tokens, batch_max_new_tokens, max_score, end_time)

                for subkey_id, scored_beams in dfs_result:

                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:

                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=hash(bk) % 1024**2)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores
                        
                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")


In [ ]:
%%writefile starter.py
import os
import time
import json
import torch
import argparse
import torch.multiprocessing as mp

# Order the task queue by estimated cost ascending instead of alphabetically.
# Unfinished tasks score 0 (coverage is the binding constraint), so finishing
# cheap tasks first maximizes completed-task count for the same wall clock.
# False -> exactly the baseline sorted-key order.
CHEAP_FIRST_ORDER = True

# Read Leg C verified-induction results (if the pre-pass ran) and refund their
# queue slots to the base solver. No results file -> empty skip set -> baseline
# behavior. Must match LEGC_ENABLED in the Leg C launch cell.
LEGC_ENABLED = True


def task_cost(task):
    """Estimated serialized token cost of one task (pure arithmetic).

    Each h x w grid costs h*w digit tokens + h newline/end tokens; a task
    costs the sum over train pair inputs+outputs and test inputs.
    """
    def grid_tokens(grid):
        return len(grid) * len(grid[0]) + len(grid)

    cost = 0
    for pair in task["train"]:
        cost += grid_tokens(pair["input"]) + grid_tokens(pair["output"])
    for pair in task["test"]:
        cost += grid_tokens(pair["input"])
    return cost


def order_keys(data, cheap_first):
    """Deterministic queue order; ties broken by key either way."""
    if not cheap_first:
        return sorted(data.keys())
    return sorted(data.keys(), key=lambda k: (task_cost(data[k]), k))


def fully_verified_task_ids(results):
    """Return tasks safe to remove from the base queue.

    Leg C may verify only a subset of a multi-test task's outputs.  Skipping
    such a task would discard the base solver's predictions for the remaining
    outputs, so queue exclusion requires a non-empty verified entry for every
    output position.
    """
    result = set()
    for task_id, record in results.items():
        if not isinstance(record, dict) or not record.get("verified"):
            continue
        outputs = record.get("outputs")
        if (
            isinstance(outputs, list)
            and outputs
            and all(
                isinstance(output, dict) and output.get("attempt") is not None
                for output in outputs
            )
        ):
            result.add(task_id)
    return result


def local_worker(rank, queue, end_time):
    
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    # Fix Unsloth patching issue
    if rank > 0:
        while not os.path.exists(f"/kaggle/worker{rank-1}"):
            time.sleep(5)
    
    from arc_solver import worker

    with open(f"/kaggle/worker{rank}", "w") as f:
        f.write("Ok")
    
    print(f"[Rank {rank}] start!")
    
    worker(rank, queue, end_time)
    
    print(f"[Rank {rank}] done!")


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    legc_skip = set()
    legc_path = "/kaggle/working/induction_results.json"
    if LEGC_ENABLED and os.path.exists(legc_path):
        with open(legc_path) as f:
            legc_skip = fully_verified_task_ids(json.load(f))
        print(f"[LegC] skipping {len(legc_skip)} verified task(s) in the base queue")

    # Exclude verified tasks BEFORE ordering so the refunded time goes to the
    # cheap-first frontier of the remaining tasks.
    remaining = {k: v for k, v in data.items() if k not in legc_skip}

    queue = mp.Manager().Queue()

    for key in order_keys(remaining, CHEAP_FIRST_ORDER):
        if not rerun_mode:
            if key not in ["0934a4d8", "36a08778", "981571dc", "aa4ec2a5"]:
                continue
        queue.put(key)
    for _ in range(4):
        queue.put(None)
    
    mp.spawn(local_worker, args=(queue, args.end_time), nprocs=4)


In [ ]:
%%writefile probe_core.py
"""Evaluation core for the isolated Nemotron Lightning ARC probe.

The model writes a Python ``transform(grid)`` function.  Candidate programs run
in a short-lived subprocess, are checked on every demonstration, and only
demo-verified programs are allowed to predict the held-out test grid.

This is an evaluation instrument, not a competition submission solver.
"""

from __future__ import annotations

import argparse
import ast
import builtins
import hashlib
import json
import os
import re
import subprocess
import sys
import tempfile
from collections import Counter
from math import exp, fsum, log
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterable


Grid = list[list[int]]

SYSTEM_PROMPT = """You are an expert at solving ARC-AGI (Abstraction and Reasoning Corpus) puzzles by writing Python code.
Your goal is to analyze input-output examples and create a `transform` function that correctly transforms any input grid into the corresponding output grid.

Analyze grid dimensions, colors, objects, symmetries, spatial operations, and patterns. Find the simplest single rule that works for ALL examples.

Code requirements:
- Function signature: `def transform(grid: list[list[int]]) -> list[list[int]]:`
- Input is a rectangular 2D list of integers 0-9.
- Return a rectangular 2D list of integers 0-9.
- Wrap code in ```python ... ```.
- Available imports: numpy, scipy, itertools, collections.
- The function must generalize to new valid grid sizes.
- Do not read files, use the network, start processes, or include an `if __name__` block.

Before the code, briefly explain the rule and why it fits every example."""

MINIMAL_SYSTEM_PROMPT = """Solve the ARC-AGI task from the demonstrations.
Infer a general transformation and return a Python function with signature
`def transform(grid: list[list[int]]) -> list[list[int]]:`. Explain your rule
briefly, then put the function in a ```python``` block. The input and output are
rectangular grids with integer colors 0-9. Do not use files, network access,
hidden-test data, or an `if __name__` block."""

FREEFORM_SYSTEM_PROMPT = """Work out the ARC-AGI transformation in the way you find most reliable.
Explore any relevant spatial, object, relational, color, or compositional
interpretation, and then provide a general Python `transform(grid)` function.
Return a brief explanation followed by the function in a ```python``` block.
The function receives and returns rectangular list-of-list grids with colors
0-9; do not use files, network access, hidden-test data, or an `if __name__`
block."""

PROMPT_STYLES = {
    "strict": SYSTEM_PROMPT,
    "minimal": MINIMAL_SYSTEM_PROMPT,
    "freeform": FREEFORM_SYSTEM_PROMPT,
}


@dataclass(frozen=True)
class CandidateResult:
    program_hash: str
    status: str
    pairs_passed: int = 0
    n_pairs: int = 0
    prediction: Grid | None = None
    error: str | None = None
    first_failed_demo: int | None = None
    observed: Grid | None = None
    expected: Grid | None = None


def render_grid(grid: Grid) -> str:
    """Render a grid in NVIDIA's compact prompt representation."""
    return "\n".join("".join(str(cell) for cell in row) for row in grid)


def build_user_prompt(train: list[dict[str, Grid]], test_input: Grid) -> str:
    chunks = ["Please solve this ARC-AGI problem:\n"]
    for index, pair in enumerate(train, 1):
        chunks.append(
            f"Train Example {index}:\n\nInput:\n{render_grid(pair['input'])}"
            f"\n\nOutput:\n{render_grid(pair['output'])}\n"
        )
    chunks.append(f"\nTest Input:\n{render_grid(test_input)}\n")
    return "\n".join(chunks)


def build_messages(
    train: list[dict[str, Grid]],
    test_input: Grid,
    *,
    prompt_style: str = "strict",
) -> list[dict[str, str]]:
    if prompt_style not in PROMPT_STYLES:
        raise ValueError(f"unknown prompt style: {prompt_style}")
    return [
        {"role": "system", "content": PROMPT_STYLES[prompt_style]},
        {"role": "user", "content": build_user_prompt(train, test_input)},
    ]


_FENCE_RE = re.compile(r"```(?:python|py)?\s*(.*?)```", re.IGNORECASE | re.DOTALL)


def extract_program(text: str) -> str | None:
    """Extract the first fenced candidate containing ``def transform``."""
    for block in _FENCE_RE.findall(text or ""):
        if re.search(r"\bdef\s+transform\s*\(", block):
            return block.strip()
    match = re.search(r"\bdef\s+transform\s*\(", text or "")
    if match:
        return text[match.start():].strip()
    return None


def program_hash(source: str) -> str:
    normalized = "\n".join(line.rstrip() for line in source.strip().splitlines())
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()[:16]


_ALLOWED_IMPORT_ROOTS = {"numpy", "scipy", "itertools", "collections"}
_BANNED_NODES = (
    ast.AsyncFunctionDef,
    ast.AsyncFor,
    ast.AsyncWith,
    ast.Await,
    ast.ClassDef,
    ast.Global,
    ast.Nonlocal,
    ast.With,
)
_BANNED_NAMES = {
    "breakpoint", "compile", "eval", "exec", "exit", "getattr", "globals",
    "help", "input", "locals", "memoryview", "open", "quit", "setattr",
    "vars", "__import__",
}
_BANNED_ATTRIBUTES = {
    "ctypes", "dump", "dumps", "fromfile", "load", "loads", "memmap",
    "popen", "save", "savetxt", "system", "tofile",
}
_MAX_SOURCE_CHARS = 32_000
_MAX_AST_NODES = 6_000


def validate_program(source: str) -> str | None:
    if len(source) > _MAX_SOURCE_CHARS:
        return f"source exceeds {_MAX_SOURCE_CHARS} character budget"
    try:
        tree = ast.parse(source)
    except SyntaxError as exc:
        return f"syntax error: {exc}"
    if sum(1 for _ in ast.walk(tree)) > _MAX_AST_NODES:
        return f"program exceeds {_MAX_AST_NODES} AST-node budget"
    transforms = [
        node for node in tree.body
        if isinstance(node, ast.FunctionDef) and node.name == "transform"
    ]
    if len(transforms) != 1:
        return "candidate must define exactly one top-level transform function"
    for node in ast.walk(tree):
        if isinstance(node, _BANNED_NODES):
            return f"banned construct: {type(node).__name__}"
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            modules = (
                [alias.name for alias in node.names]
                if isinstance(node, ast.Import)
                else [node.module or ""]
            )
            if any(module.split(".", 1)[0] not in _ALLOWED_IMPORT_ROOTS for module in modules):
                return f"banned import: {', '.join(modules)}"
        if isinstance(node, ast.Name) and node.id in _BANNED_NAMES:
            return f"banned name: {node.id}"
        if isinstance(node, ast.Attribute):
            if node.attr.startswith("__") or node.attr in _BANNED_ATTRIBUTES:
                return f"banned attribute: {node.attr}"
    return None


def _valid_grid(value: Any) -> bool:
    if not isinstance(value, list) or not 1 <= len(value) <= 30:
        return False
    if not all(isinstance(row, list) for row in value):
        return False
    width = len(value[0]) if value else 0
    if not 1 <= width <= 30 or any(len(row) != width for row in value):
        return False
    return all(
        isinstance(cell, int) and not isinstance(cell, bool) and 0 <= cell <= 9
        for row in value for cell in row
    )


def _normalize_grid(value: Any) -> Grid:
    if hasattr(value, "tolist"):
        value = value.tolist()
    if not _valid_grid(value):
        raise ValueError("transform returned an invalid ARC grid")
    return [[int(cell) for cell in row] for row in value]


def _safe_import(
    name: str,
    globals_: dict[str, Any] | None = None,
    locals_: dict[str, Any] | None = None,
    fromlist: tuple[str, ...] = (),
    level: int = 0,
) -> Any:
    if level or name.split(".", 1)[0] not in _ALLOWED_IMPORT_ROOTS:
        raise ImportError(f"import of {name!r} is disabled")
    return builtins.__import__(name, globals_, locals_, fromlist, level)


def _safe_builtins() -> dict[str, Any]:
    names = (
        "abs", "all", "any", "bool", "dict", "divmod", "enumerate", "filter",
        "float", "frozenset", "int", "isinstance", "len", "list", "map", "max",
        "min", "pow", "range", "repr", "reversed", "round", "set", "slice",
        "sorted", "str", "sum", "tuple", "zip", "Exception", "ValueError",
    )
    result = {name: getattr(builtins, name) for name in names}
    result["__import__"] = _safe_import
    return result


def _limit_worker(cpu_seconds: int, memory_gib: int) -> None:
    """Apply Linux resource limits when available; subprocess timeout is the backstop."""
    try:
        import resource

        memory = memory_gib * 1024**3
        resource.setrlimit(resource.RLIMIT_CPU, (cpu_seconds, cpu_seconds + 1))
        resource.setrlimit(resource.RLIMIT_AS, (memory, memory))
        resource.setrlimit(resource.RLIMIT_FSIZE, (1_000_000, 1_000_000))
        resource.setrlimit(resource.RLIMIT_NOFILE, (32, 32))
    except (ImportError, OSError, ValueError):
        pass


def _worker(request_path: str, response_path: str) -> int:
    request = json.loads(Path(request_path).read_text(encoding="utf-8"))
    source = request["source"]
    error = validate_program(source)
    if error:
        result = {"status": "rejected", "pairs_passed": 0, "error": error}
    else:
        _limit_worker(int(request["cpu_seconds"]), int(request["memory_gib"]))
        namespace: dict[str, Any] = {"__builtins__": _safe_builtins()}
        try:
            exec(compile(source, "<arc-candidate>", "exec"), namespace)
            transform = namespace["transform"]
            passed = 0
            first_error = None
            first_failed_demo = None
            first_observed = None
            first_expected = None
            for index, pair in enumerate(request["train"]):
                try:
                    got = _normalize_grid(transform([row[:] for row in pair["input"]]))
                    if got == pair["output"]:
                        passed += 1
                    elif first_error is None:
                        first_error = f"demo {index} mismatch"
                        first_failed_demo = index
                        first_observed = got
                        first_expected = pair["output"]
                except Exception as exc:  # candidate failure, contained in worker
                    if first_error is None:
                        first_error = f"demo {index}: {type(exc).__name__}: {exc}"
                        first_failed_demo = index
                        first_expected = pair["output"]
            if passed == len(request["train"]):
                prediction = _normalize_grid(
                    transform([row[:] for row in request["test_input"]])
                )
                result = {
                    "status": "verified",
                    "pairs_passed": passed,
                    "n_pairs": len(request["train"]),
                    "prediction": prediction,
                }
            else:
                result = {
                    "status": "partial",
                    "pairs_passed": passed,
                    "n_pairs": len(request["train"]),
                    "error": first_error,
                    "first_failed_demo": first_failed_demo,
                    "observed": first_observed,
                    "expected": first_expected,
                }
        except BaseException as exc:  # worker is disposable
            result = {
                "status": "error",
                "pairs_passed": 0,
                "n_pairs": len(request["train"]),
                "error": f"{type(exc).__name__}: {exc}"[:400],
            }
    Path(response_path).write_text(json.dumps(result), encoding="utf-8")
    return 0


def run_candidate(
    source: str,
    train: list[dict[str, Grid]],
    test_input: Grid,
    timeout_seconds: float = 8.0,
    memory_gib: int = 4,
) -> CandidateResult:
    digest = program_hash(source)
    static_error = validate_program(source)
    if static_error:
        return CandidateResult(digest, "rejected", error=static_error)
    request = {
        "source": source,
        "train": train,
        "test_input": test_input,
        "cpu_seconds": max(1, int(timeout_seconds)),
        "memory_gib": memory_gib,
    }
    with tempfile.TemporaryDirectory(prefix="arc_nemotron_") as temp_dir:
        request_path = Path(temp_dir) / "request.json"
        response_path = Path(temp_dir) / "response.json"
        request_path.write_text(json.dumps(request), encoding="utf-8")
        try:
            completed = subprocess.run(
                [sys.executable, str(Path(__file__).resolve()), "_worker",
                 str(request_path), str(response_path)],
                capture_output=True,
                text=True,
                timeout=timeout_seconds + 3.0,
                check=False,
            )
        except subprocess.TimeoutExpired:
            return CandidateResult(digest, "timeout", error="subprocess timeout")
        if completed.returncode != 0 or not response_path.exists():
            detail = (completed.stderr or completed.stdout or "worker failed")[-400:]
            return CandidateResult(digest, "error", error=detail)
        raw = json.loads(response_path.read_text(encoding="utf-8"))
    return CandidateResult(
        program_hash=digest,
        status=raw["status"],
        pairs_passed=int(raw.get("pairs_passed", 0)),
        n_pairs=int(raw.get("n_pairs", len(train))),
        prediction=raw.get("prediction"),
        error=raw.get("error"),
        first_failed_demo=raw.get("first_failed_demo"),
        observed=raw.get("observed"),
        expected=raw.get("expected"),
    )


def grid_key(grid: Grid) -> str:
    return json.dumps(grid, separators=(",", ":"))


def rank_verified_outputs(
    results: Iterable[CandidateResult],
    *,
    correlation_groups: dict[str, str] | None = None,
    mdl_lengths: dict[str, float] | None = None,
    collapse_correlated: bool = False,
) -> tuple[tuple[str, float, tuple[str, ...]], ...]:
    """Rank verified output classes with optional lineage correction.

    The default keeps each program hash independent for backwards-compatible
    probe behavior.  In collapse mode, one lineage contributes one normalized
    semantic distribution; within a lineage/output class the shortest known
    program supplies the MDL weight.
    """

    verified = tuple(
        result for result in results
        if result.status == "verified" and result.prediction is not None
    )
    if not verified:
        return ()
    groups: dict[str, list[CandidateResult]] = {}
    for result in verified:
        group = (
            (correlation_groups or {}).get(result.program_hash, result.program_hash)
            if collapse_correlated else result.program_hash
        )
        groups.setdefault(group, []).append(result)
    class_mass: dict[str, float] = {}
    witness_groups: dict[str, set[str]] = {}
    group_weight = 1.0 / len(groups)
    lengths = mdl_lengths or {}
    for group, members in sorted(groups.items()):
        by_output: dict[str, list[CandidateResult]] = {}
        for member in members:
            by_output.setdefault(grid_key(member.prediction), []).append(member)
        scores = {
            output: max(exp(-lengths.get(member.program_hash, 0.0) * log(2.0))
                        for member in output_members)
            for output, output_members in by_output.items()
        }
        total = fsum(scores.values())
        for output, score in scores.items():
            class_mass[output] = class_mass.get(output, 0.0) + group_weight * score / total
            witness_groups.setdefault(output, set()).add(group)
    ranked = sorted(class_mass, key=lambda output: (-class_mass[output], output))
    return tuple(
        (output, class_mass[output], tuple(sorted(witness_groups[output])))
        for output in ranked
    )


def evaluate_responses(
    responses: Iterable[str],
    train: list[dict[str, Grid]],
    test_input: Grid,
    truth: Grid | None = None,
    timeout_seconds: float = 8.0,
    correlation_groups: dict[str, str] | None = None,
    mdl_lengths: dict[str, float] | None = None,
    collapse_correlated: bool = False,
) -> dict[str, Any]:
    responses = list(responses)
    programs: dict[str, str] = {}
    parse_count = 0
    for text in responses:
        source = extract_program(text)
        if source is not None:
            parse_count += 1
            programs.setdefault(program_hash(source), source)

    results = [
        run_candidate(source, train, test_input, timeout_seconds)
        for source in programs.values()
    ]
    verified = [result for result in results if result.status == "verified"]
    votes = Counter(grid_key(result.prediction) for result in verified if result.prediction)
    ranked_classes = rank_verified_outputs(
        results,
        correlation_groups=correlation_groups,
        mdl_lengths=mdl_lengths,
        collapse_correlated=collapse_correlated,
    )
    ranked = [json.loads(key) for key, _, _ in ranked_classes]
    correct_verified = (
        sum(1 for result in verified if result.prediction == truth)
        if truth is not None else None
    )
    return {
        "n_responses": len(responses),
        "n_parsed": parse_count,
        "n_unique_programs": len(programs),
        "n_verified_programs": len(verified),
        "status_counts": dict(Counter(result.status for result in results)),
        "n_unique_verified_outputs": len(votes),
        "top_predictions": ranked[:2],
        "top_vote_counts": [count for _, count in votes.most_common(2)],
        "top_class_masses": [mass for _, mass, _ in ranked_classes[:2]],
        "top_class_witness_groups": [groups for _, _, groups in ranked_classes[:2]],
        "oracle_correct": (
            any(result.prediction == truth for result in verified)
            if truth is not None else None
        ),
        "top1_correct": ranked[0] == truth if truth is not None and ranked else False,
        "top2_correct": (
            any(prediction == truth for prediction in ranked[:2])
            if truth is not None else None
        ),
        "correct_verified_programs": correct_verified,
        "candidates": [asdict(result) for result in results],
    }


def aggregate_records(records: list[dict[str, Any]]) -> dict[str, Any]:
    responses = sum(record["n_responses"] for record in records)
    parsed = sum(record["n_parsed"] for record in records)
    unique = sum(record["n_unique_programs"] for record in records)
    verified = sum(record["n_verified_programs"] for record in records)
    covered = sum(bool(record["n_verified_programs"]) for record in records)
    total = len(records)
    return {
        "outputs": total,
        "responses": responses,
        "parse_rate": parsed / responses if responses else 0.0,
        "unique_program_rate": unique / parsed if parsed else 0.0,
        "demo_verified_program_rate": verified / unique if unique else 0.0,
        "verified_output_coverage": covered / total if total else 0.0,
        "oracle_pass_at_k": sum(bool(r["oracle_correct"]) for r in records) / total if total else 0.0,
        "top1_accuracy": sum(bool(r["top1_correct"]) for r in records) / total if total else 0.0,
        "top2_accuracy": sum(bool(r["top2_correct"]) for r in records) / total if total else 0.0,
    }


def _main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("mode", choices=["_worker", "selftest"])
    parser.add_argument("paths", nargs="*")
    args = parser.parse_args()
    if args.mode == "_worker":
        if len(args.paths) != 2:
            parser.error("_worker requires request and response paths")
        return _worker(args.paths[0], args.paths[1])

    train = [
        {"input": [[0, 2], [0, 0]], "output": [[0, 3], [0, 0]]},
        {"input": [[2, 0], [0, 2]], "output": [[3, 0], [0, 3]]},
    ]
    good = """```python
def transform(grid):
    return [[3 if cell == 2 else cell for cell in row] for row in grid]
```"""
    bad = """```python
import os
def transform(grid):
    return grid
```"""
    report = evaluate_responses([good, bad], train, [[2]], [[3]])
    assert report["n_parsed"] == 2
    assert report["n_verified_programs"] == 1
    assert report["top1_correct"] is True
    print(json.dumps(report, indent=2))
    print("Nemotron probe selftest: PASS")
    return 0


if __name__ == "__main__":
    raise SystemExit(_main())


In [ ]:
%%writefile nemotron_induction.py
"""Nemotron Leg C: verified program induction over the real competition queue.

Same contract as arc_induction_v2.py's output (Leg C merge cell in the base
notebook consumes this unchanged): writes /kaggle/working/induction_results.json
as {task_id: {"verified": bool, "outputs": [{"attempt": grid|None, "alt": grid|None}, ...]}}.

Engine and verification are the tested probe_core.py functions (prompt format,
AST-sandboxed demo-only verification, majority vote); this script points them at
the real test file (or the smoke subset) instead of a diagnostic dev sample, and
generates in cost-ascending, deadline-checked chunks so a budget overrun degrades
to partial coverage instead of a hard failure.  The default is one sampling
batch and legacy raw-majority behavior; ``--lineages`` and ``--lineage-aware``
are explicit shadow-fold controls for correlation-aware sampling.
"""

from __future__ import annotations

import argparse
import json
import os
import time
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

try:  # package import in local tests; flat import in the generated Kaggle notebook
    from .probe_core import (
        CandidateResult,
        build_messages,
        extract_program,
        grid_key,
        program_hash,
        rank_verified_outputs,
        run_candidate,
    )
except ImportError:  # pragma: no cover - exercised inside the Kaggle working dir
    from probe_core import (
        CandidateResult,
        build_messages,
        extract_program,
        grid_key,
        program_hash,
        rank_verified_outputs,
        run_candidate,
    )


@dataclass(frozen=True)
class SamplingLineage:
    """One independently seeded decoding group in a fixed sample budget."""

    name: str
    sample_count: int
    seed: int
    temperature: float
    prompt_style: str = "strict"


def make_sampling_lineages(
    total_samples: int,
    lineage_count: int,
    *,
    seed: int = 260901,
    temperatures: tuple[float, ...] = (1.0,),
    prompt_styles: tuple[str, ...] = ("strict",),
) -> tuple[SamplingLineage, ...]:
    """Return a balanced, deterministic lineage plan.

    ``lineage_count=1`` and the default temperature reproduce the historical
    one-batch contract.  Extra lineages split the same total sample count;
    callers may provide a frozen temperature schedule for a shadow-fold
    ablation.  The runner never infers independence from the names: the
    resulting IDs are carried explicitly to the verifier.
    """

    if total_samples < 1 or lineage_count < 1:
        raise ValueError("sample and lineage counts must be positive")
    if lineage_count > total_samples:
        raise ValueError("lineage_count cannot exceed total_samples")
    if not temperatures or any(temperature <= 0.0 for temperature in temperatures):
        raise ValueError("temperatures must be non-empty and positive")
    if not prompt_styles or any(style not in {"strict", "minimal", "freeform"}
                                for style in prompt_styles):
        raise ValueError("prompt_styles must be non-empty known styles")
    quotient, remainder = divmod(total_samples, lineage_count)
    return tuple(
        SamplingLineage(
            name=f"lineage-{index}",
            sample_count=quotient + (1 if index < remainder else 0),
            seed=seed + 1009 * index,
            temperature=temperatures[index % len(temperatures)],
            prompt_style=prompt_styles[index % len(prompt_styles)],
        )
        for index in range(lineage_count)
    )


def find_test_path() -> Path:
    root = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2")
    name = "arc-agi_test_challenges.json" if os.getenv("KAGGLE_IS_COMPETITION_RERUN") \
        else "arc-agi_evaluation_challenges.json"
    path = root / name
    if not path.exists():
        raise FileNotFoundError(f"competition challenges file missing: {path}")
    return path


def find_model(override: str | None) -> str:
    if override and Path(override).exists():
        return override
    for root in (Path("/kaggle/input"), Path("/kaggle/working")):
        if not root.exists():
            continue
        for config in root.rglob("config.json"):
            low = str(config.parent).lower()
            if "nemotron" in low and "lightning" in low:
                return str(config.parent)
    raise FileNotFoundError("Nemotron Lightning model not attached; pass --model-path")


def prompt_cost(unit: dict[str, Any]) -> int:
    grids = [grid for pair in unit["train"] for grid in (pair["input"], pair["output"])]
    grids.append(unit["test_input"])
    return sum(len(grid) * len(grid[0]) for grid in grids)


def load_units(challenges: dict[str, Any]) -> list[dict[str, Any]]:
    units = []
    for task_id, task in challenges.items():
        for test_index, item in enumerate(task["test"]):
            units.append({
                "task_id": task_id,
                "test_index": test_index,
                "train": task["train"],
                "test_input": item["input"],
            })
    units.sort(key=lambda unit: (prompt_cost(unit), unit["task_id"], unit["test_index"]))
    return units


def load_engine(model_path: str, max_model_len: int, gpu_memory: float):
    import torch
    from vllm import LLM

    print(f"[induction/env] torch={torch.__version__} cuda={torch.version.cuda}", flush=True)
    print(f"[induction/env] devices={torch.cuda.device_count()}", flush=True)
    kwargs = dict(
        model=model_path,
        tensor_parallel_size=max(1, torch.cuda.device_count()),
        quantization="modelopt_fp4",
        trust_remote_code=True,
        max_model_len=max_model_len,
        gpu_memory_utilization=gpu_memory,
        enforce_eager=True,
        enable_prefix_caching=True,
    )
    print("[induction/engine] START", json.dumps(kwargs, default=str), flush=True)
    started = time.perf_counter()
    engine = LLM(**kwargs)
    print(f"[induction/engine] READY boot_seconds={time.perf_counter() - started:.1f}", flush=True)
    return engine


def verify_unit(
    unit: dict[str, Any],
    texts: list[str],
    timeout_seconds: float,
    *,
    lineage_texts: tuple[tuple[str, list[str]], ...] | None = None,
    collapse_correlated: bool = False,
    include_diagnostics: bool = False,
    diagnostic_limit: int = 8,
    diagnostic_trace_chars: int = 6_000,
) -> dict[str, Any]:
    """Verify one unit and rank outputs, optionally by decoding lineage.

    The legacy ``texts`` path remains raw-majority compatible.  The opt-in
    lineage path keeps one normalized output distribution per lineage and then
    ranks semantic output classes through ``probe_core``.
    """

    if diagnostic_limit < 0:
        raise ValueError("diagnostic_limit must be non-negative")
    if diagnostic_trace_chars < 0:
        raise ValueError("diagnostic_trace_chars must be non-negative")
    diagnostics: list[dict[str, Any]] = []

    def finish(value: dict[str, Any]) -> dict[str, Any]:
        if include_diagnostics:
            value["diagnostics"] = diagnostics[:diagnostic_limit]
        return value

    def record(
        lineage: str,
        source: str,
        raw_text: str,
        result: CandidateResult,
    ) -> None:
        if not include_diagnostics or result.status == "verified":
            return
        if len(diagnostics) >= diagnostic_limit:
            return
        diagnostics.append({
            "lineage": lineage,
            "source": source,
            "result": asdict(result),
            "trace": raw_text[:diagnostic_trace_chars],
        })

    if lineage_texts is None:
        lineage_texts = (("lineage-0", texts),)
    if not lineage_texts:
        return finish({"attempt": None, "alt": None})

    if not collapse_correlated:
        flattened = [text for _, group_texts in lineage_texts for text in group_texts]
        programs: dict[str, tuple[str, str]] = {}
        for text in flattened:
            source = extract_program(text)
            if source is not None:
                programs.setdefault(program_hash(source), (source, text))
        predictions = []
        for source, raw_text in programs.values():
            result = run_candidate(source, unit["train"], unit["test_input"], timeout_seconds)
            record("lineage-0", source, raw_text, result)
            if result.status == "verified" and result.prediction is not None:
                predictions.append(result.prediction)
        if not predictions:
            return finish({"attempt": None, "alt": None})
        votes = Counter(grid_key(g) for g in predictions)
        ranked = [json.loads(key) for key, _ in votes.most_common()]
        return finish({"attempt": ranked[0], "alt": ranked[1] if len(ranked) > 1 else None})

    verified: list[CandidateResult] = []
    correlation_groups: dict[str, str] = {}
    for lineage, group_texts in lineage_texts:
        programs: dict[str, tuple[str, str]] = {}
        for text in group_texts:
            source = extract_program(text)
            if source is not None:
                programs.setdefault(program_hash(source), (source, text))
        for source, raw_text in programs.values():
            result = run_candidate(source, unit["train"], unit["test_input"], timeout_seconds)
            record(lineage, source, raw_text, result)
            if result.status == "verified" and result.prediction is not None:
                unique_id = f"{result.program_hash}@{lineage}"
                verified.append(CandidateResult(
                    program_hash=unique_id,
                    status=result.status,
                    pairs_passed=result.pairs_passed,
                    n_pairs=result.n_pairs,
                    prediction=result.prediction,
                    error=result.error,
                ))
                correlation_groups[unique_id] = lineage

    ranked = rank_verified_outputs(
        verified,
        correlation_groups=correlation_groups,
        collapse_correlated=True,
    )
    if not ranked:
        return finish({"attempt": None, "alt": None})
    outputs = [json.loads(key) for key, _, _ in ranked]
    return finish({"attempt": outputs[0], "alt": outputs[1] if len(outputs) > 1 else None})


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--model-path")
    parser.add_argument("--k", type=int, default=6)
    parser.add_argument("--seed", type=int, default=260901)
    parser.add_argument("--lineages", type=int, default=1,
                        help="balanced independent decoding groups (default: 1)")
    parser.add_argument("--lineage-temperatures", default="1.0",
                        help="comma-separated frozen temperatures for lineages")
    parser.add_argument("--prompt-styles", default="strict",
                        help="comma-separated prompt styles for lineages")
    parser.add_argument("--lineage-aware", action="store_true",
                        help="collapse verified outputs by decoding lineage")
    parser.add_argument("--diagnostics", action="store_true",
                        help="persist bounded failed-candidate repair telemetry")
    parser.add_argument("--diagnostic-limit", type=int, default=8,
                        help="maximum failed candidates recorded per output")
    parser.add_argument("--diagnostic-trace-chars", type=int, default=6000,
                        help="maximum raw trace characters recorded per candidate")
    parser.add_argument("--max-new-tokens", type=int, default=2048)
    parser.add_argument("--max-model-len", type=int, default=16384)
    parser.add_argument("--gpu-memory", type=float, default=0.88)
    parser.add_argument("--program-timeout", type=float, default=8.0)
    parser.add_argument("--chunk-size", type=int, default=16)
    parser.add_argument("--budget-h", type=float, default=1.0)
    parser.add_argument("--end-ts", type=float, default=0.0)
    parser.add_argument("--tasks", default="", help="comma-separated task-id allowlist (smoke mode)")
    parser.add_argument("--output", default="/kaggle/working/induction_results.json")
    parser.add_argument("--selftest", action="store_true")
    args = parser.parse_args()

    if args.selftest:
        from probe_core import _main as probe_selftest  # noqa: F401
        import subprocess
        import sys
        return subprocess.run([sys.executable, "probe_core.py", "selftest"], check=False).returncode

    deadline = args.end_ts if args.end_ts else (time.time() + args.budget_h * 3600)

    challenges = json.loads(find_test_path().read_text(encoding="utf-8"))
    if args.tasks:
        allow = {t for t in args.tasks.split(",") if t}
        challenges = {k: v for k, v in challenges.items() if k in allow}
    units = load_units(challenges)
    print(f"[induction] tasks={len(challenges)} outputs={len(units)} k={args.k} "
          f"deadline_in_min={max(0.0, deadline - time.time()) / 60:.1f}", flush=True)

    results: dict[str, dict[str, Any]] = {}
    for task_id, task in challenges.items():
        results[task_id] = {"verified": False, "outputs": [{"attempt": None, "alt": None}
                                                             for _ in task["test"]]}

    if time.time() > deadline:
        print("[induction] no budget remaining before engine boot; writing empty results", flush=True)
        Path(args.output).write_text(json.dumps(results), encoding="utf-8")
        return 0

    model_path = find_model(args.model_path)
    engine = load_engine(model_path, args.max_model_len, args.gpu_memory)
    from vllm import SamplingParams

    tokenizer = engine.get_tokenizer()
    temperatures = tuple(float(value) for value in args.lineage_temperatures.split(",") if value.strip())
    prompt_styles = tuple(value.strip() for value in args.prompt_styles.split(",") if value.strip())
    lineages = make_sampling_lineages(
        args.k,
        args.lineages,
        seed=args.seed,
        temperatures=temperatures,
        prompt_styles=prompt_styles,
    )
    print("[induction] lineages:", json.dumps([lineage.__dict__ for lineage in lineages]), flush=True)

    processed = verified_count = 0
    for start in range(0, len(units), args.chunk_size):
        if time.time() > deadline:
            print(f"[induction] budget exhausted after {processed}/{len(units)} outputs", flush=True)
            break
        chunk = units[start:start + args.chunk_size]
        lineage_outputs: list[list[tuple[str, list[str]]]] = [[] for _ in chunk]
        started = time.perf_counter()
        for lineage in lineages:
            prompts = [
                tokenizer.apply_chat_template(
                    build_messages(u["train"], u["test_input"],
                                   prompt_style=lineage.prompt_style),
                    tokenize=False,
                    add_generation_prompt=True,
                )
                for u in chunk
            ]
            sampling = SamplingParams(
                n=lineage.sample_count,
                temperature=lineage.temperature,
                top_p=0.95,
                max_tokens=args.max_new_tokens,
                seed=lineage.seed,
            )
            outputs = engine.generate(prompts, sampling, use_tqdm=False)
            for index, output in enumerate(outputs):
                lineage_outputs[index].append(
                    (lineage.name, [sample.text for sample in output.outputs])
                )
        elapsed = time.perf_counter() - started
        print(f"[induction] chunk {start}-{start + len(chunk)} generated in {elapsed:.1f}s", flush=True)

        for unit, groups in zip(chunk, lineage_outputs):
            entry = verify_unit(
                unit,
                [text for _, texts in groups for text in texts],
                args.program_timeout,
                lineage_texts=tuple(groups),
                collapse_correlated=args.lineage_aware,
                include_diagnostics=args.diagnostics,
                diagnostic_limit=args.diagnostic_limit,
                diagnostic_trace_chars=args.diagnostic_trace_chars,
            )
            rec = results[unit["task_id"]]
            rec["outputs"][unit["test_index"]] = entry
            if entry["attempt"] is not None:
                rec["verified"] = True
                verified_count += 1
            processed += 1
            print(f"[induction/result] {unit['task_id']}#{unit['test_index']} "
                  f"verified={entry['attempt'] is not None}", flush=True)

    Path(args.output).parent.mkdir(parents=True, exist_ok=True)
    Path(args.output).write_text(json.dumps(results), encoding="utf-8")
    n_tasks_verified = sum(1 for r in results.values() if r["verified"])
    print(f"[induction] wrote {args.output}: {n_tasks_verified}/{len(results)} tasks, "
          f"{verified_count}/{processed} outputs verified ({processed}/{len(units)} attempted)",
          flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
# CPU-only safety and metric smoke test for the induction verifier.
import subprocess, sys
subprocess.run([sys.executable, 'probe_core.py', 'selftest'], check=True)


In [ ]:
# Install the attached offline vLLM 0.27 CUDA wheelhouse into an isolated
# target -- NOT the base site-packages, and NOT the kernel's os.environ, so
# it cannot leak into the unsloth-based base pipeline's own subprocess.
import pathlib, subprocess, sys
wheel_candidates = [
    pathlib.Path('/kaggle/input/vllm-027-cuda-wheels'),
    pathlib.Path('/kaggle/input/datasets/vladimiryakunin/vllm-027-cuda-wheels'),
]
vllm_wheelhouse = next((p for p in wheel_candidates if (p / 'requirements.lock').is_file()), None)
if vllm_wheelhouse is None:
    raise FileNotFoundError(f'offline vLLM wheelhouse missing: {wheel_candidates}')
vllm_target = pathlib.Path('/kaggle/working/vllm-site-packages')
vllm_target.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index',
    '--find-links', str(vllm_wheelhouse), '--requirement', str(vllm_wheelhouse / 'requirements.lock'),
    '--target', str(vllm_target), '--upgrade', '--ignore-installed',
    '--only-binary', ':all:', '--no-compile', '--disable-pip-version-check',
    '--no-warn-conflicts'], check=True)
print('[induction] isolated vLLM site-packages at', vllm_target)


In [ ]:
# ===================== Leg C: verified program induction (pre-pass) =====================
# Nemotron-3.5-Lightning-30B-A3B-NVFP4 (vLLM) samples candidate transform()
# programs per task; probe_core's AST-whitelisted, subprocess-isolated sandbox
# executes them against EVERY demonstration pair. Fully verified programs'
# outputs pre-empt attempt_1 in the final merge and their tasks are removed
# from the base queue (time refund). Empty results (missing model, budget too
# small, or nothing verifies) -> starter and merge are no-ops, i.e. this
# notebook degrades exactly to the Launch B baseline.
# LEGC_ENABLED = False -> nothing runs, no results file -> byte-equivalent
# baseline behavior. Must match the flag in starter.py.
import os
import sys
import subprocess
import time

LEGC_ENABLED   = True
LEGC_BUDGET_H  = 1.0   # wall budget of the whole induction phase (public-head value)
BASE_RESERVE_H = 9.8   # wall hours guaranteed to remain for the base pipeline
# Research controls: defaults preserve the one-batch/raw-majority contract.
LEGC_LINEAGES = 1
LEGC_LINEAGE_AWARE = False
LEGC_LINEAGE_TEMPERATURES = '1.0'
LEGC_PROMPT_STYLES = 'strict'
LEGC_DIAGNOSTICS = False
LEGC_DIAGNOSTIC_TRACE_CHARS = 6000

# Same 4 smoke tasks as the base pipeline's eval mode (keeps the commit run short).
SMOKE_TASKS = "0934a4d8,36a08778,981571dc,aa4ec2a5"

if LEGC_ENABLED:
    # Sandbox selftest already ran above (CPU-only). vLLM-dependent gate next:
    # fail here, before model loading, if the isolated install is broken.
    induction_env = dict(os.environ)
    induction_env['PYTHONPATH'] = str(vllm_target) + os.pathsep + os.environ.get('PYTHONPATH', '')
    gate = subprocess.run(
        [sys.executable, '-c',
         'import torch, vllm; from packaging.version import Version; '
         "print('torch', torch.__version__, 'cuda', torch.version.cuda); "
         "print('vllm', vllm.__version__); "
         "assert Version(vllm.__version__) >= Version('0.27.1'); "
         'assert torch.cuda.device_count() == 4, torch.cuda.device_count()'],
        env=induction_env, check=False)
    if gate.returncode != 0:
        print('[LegC] vLLM dependency gate failed -- skipping Nemotron induction, '
              'base pipeline keeps the full budget', flush=True)
    else:
        # Enforce both controls: the Leg-C cap and the reserved base window.
        legc_end_ts = min(
            time.time() + LEGC_BUDGET_H * 3600,
            global_end_time - BASE_RESERVE_H * 3600,
        )
        cmd = [sys.executable, 'nemotron_induction.py',
               '--budget-h', f'{LEGC_BUDGET_H:.3f}',
               '--end-ts', f'{legc_end_ts:.0f}',
               '--lineages', str(LEGC_LINEAGES),
               '--lineage-temperatures', LEGC_LINEAGE_TEMPERATURES,
               '--prompt-styles', LEGC_PROMPT_STYLES,
               '--diagnostic-trace-chars', str(LEGC_DIAGNOSTIC_TRACE_CHARS)]
        if LEGC_LINEAGE_AWARE:
            cmd.append('--lineage-aware')
        if LEGC_DIAGNOSTICS:
            cmd.append('--diagnostics')
        if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
            cmd += ['--tasks', SMOKE_TASKS]
        print('[LegC] launch:', ' '.join(cmd), flush=True)
        subprocess.run(cmd, env=induction_env, check=False)
else:
    print('[LegC] disabled -- base pipeline keeps the full budget')


In [ ]:
!UNSLOTH_DISABLE_STATISTICS=1 TRITON_PTXAS_PATH=/usr/local/cuda/bin/ptxas OMP_NUM_THREADS=12 python starter.py --end-time {global_end_time}


In [ ]:
import os
import json
import numpy as np
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder, score_full_probmul_3, score_kgmon, score_log_evidence

rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

if rerun_mode:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json")
else:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
    data = data.load_replies("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json")

decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)

decoder.load_decoded_results("/kaggle/inference_outputs")

# ---- Fork v1: diverse second attempt (the only change vs. the LB33.89 baseline) ----
# Baseline behavior: run_selection_algo() ranks candidates with score_kgmon and the
# submission takes its top-2. Diverse mode keeps the score_kgmon top pick as attempt_1
# (identical to the baseline's attempt_1) and uses the score_full_probmul_3 top pick as
# attempt_2 whenever the two algorithms disagree, so the attempts test two scoring
# hypotheses instead of ranks 1-2 of a single ranking. When both algorithms agree on the
# winner (or probmul_3 has no differing candidate), fall back to the exact baseline top-2.
# DIVERSE_ATTEMPT_2 = False restores baseline selection exactly.
DIVERSE_ATTEMPT_2 = True
# Optional cache-calibrated alternative; False keeps probmul_3 as the control.
LOG_EVIDENCE_ATTEMPT_2 = False

def select_attempts(decoder):
    baseline = decoder.run_selection_algo()  # default = score_kgmon (baseline)
    if not DIVERSE_ATTEMPT_2:
        return baseline
    secondary_algo = score_log_evidence if LOG_EVIDENCE_ATTEMPT_2 else score_full_probmul_3
    secondary = decoder.run_selection_algo(secondary_algo)
    results = {}
    for bk, ranked in baseline.items():
        second = None
        if ranked:
            second = next((g for g in secondary.get(bk, []) if not np.array_equal(g, ranked[0])), None)
        results[bk] = ranked if second is None else [ranked[0], second]
    return results

submission = data.get_submission(select_attempts(decoder))

with open("submission.json", "w") as f:
    json.dump(submission, f)

if not rerun_mode:
    decoder.benchmark_selection_algos()
    with open("submission.json", "r") as f:
        reload_submission = json.load(f)
    print("*** Reload score:", data.validate_submission(reload_submission))


In [ ]:
# ===================== Leg C merge (run AFTER the submission cell) =====================
# Monotonic union: train-verified induction outputs pre-empt attempt_1; the base
# candidate is demoted to attempt_2 (cross-family decorrelation). Tasks without a
# verified program are left untouched. LEGC_ENABLED=False or a missing results file
# makes this a no-op.
import os
import json

if LEGC_ENABLED:
    SUB = "submission.json"
    IND = "/kaggle/working/induction_results.json"

    with open(SUB) as f:
        sub = json.load(f)
    ind = {}
    if os.path.exists(IND):
        with open(IND) as f:
            ind = json.load(f)

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    def local_score(s):
        try:
            return data.validate_submission(s)   # `data` from the submission cell
        except Exception:
            return None

    before = None if rerun_mode else local_score(sub)

    promoted = created = 0
    for tid, rec in ind.items():
        if not isinstance(rec, dict) or not rec.get("verified"):
            continue
        for ti, o in enumerate(rec.get("outputs", [])):
            g = o.get("attempt")
            if not g:
                continue
            alt = o.get("alt")
            if tid not in sub:
                sub[tid] = []
            while len(sub[tid]) <= ti:
                sub[tid].append({"attempt_1": [[0]], "attempt_2": [[0]]})
                created += 1
            e = sub[tid][ti]
            if e["attempt_1"] != g:
                demoted = e["attempt_1"]
                e["attempt_2"] = demoted if demoted != [[0]] else (alt or demoted)
                e["attempt_1"] = g
                promoted += 1
            elif alt and e.get("attempt_2") == [[0]]:
                e["attempt_2"] = alt

    with open(SUB, "w") as f:
        json.dump(sub, f)

    after = None if rerun_mode else local_score(sub)
    print(f"[LegC merge] promoted={promoted} created={created}")
    if before is not None and after is not None:
        print(f"[LegC merge] local score: base {before:.2f} -> merged {after:.2f} (unit = tasks)")
else:
    print("[LegC] merge skipped (disabled)")
